# Day 2 | 괴리율 v1 산출
## Korean Stock Undervaluation Analysis — Gap Calculation v1

**목표:**
- 시장 중앙값 PER / 섹터 평균 PER 기준 적정주가 산출
- 괴리율 v1 계산 → gap_v1_refined.csv (136개) 저장
- signal 분류

**데이터 흐름:**
```
clean_data_final.csv (133개)
    ↓
섹터별 중앙값 PER 계산
    ↓
시장 괴리율 / 섹터 괴리율 산출
    ↓
gap_v1_refined.csv (136개)
```


## 0. 환경 세팅

In [ ]:
!pip install -q git+https://github.com/FinanceData/FinanceDataReader.git

from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import FinanceDataReader as fdr
import time

BASE_PATH = '/content/drive/MyDrive/stock-analysis'

# corp_code 패딩 복구 필수 (CSV 저장 시 앞자리 00 소실 문제)
df_main = pd.read_csv(
    f'{BASE_PATH}/data/processed/clean_data_final.csv',
    dtype={'Code': str, 'corp_code': str}
)
df_main['corp_code'] = df_main['corp_code'].str.zfill(8)

print(f"✅ 로드 완료: {len(df_main)}개")
df_main.head()


## 1. 현재가 수집

In [ ]:
# FinanceDataReader로 현재가 수집 (days=2: 최신 종가 기준)
prices = {}
failed = []

for _, row in df_main.iterrows():
    code = row['Code']
    try:
        df_p = fdr.DataReader(code, pd.Timestamp.now() - pd.Timedelta(days=2))
        if len(df_p) > 0:
            prices[code] = df_p['Close'].iloc[-1]
        else:
            failed.append(code)
    except:
        failed.append(code)
    time.sleep(0.05)

df_main['price'] = df_main['Code'].map(prices)
print(f"✅ 현재가 수집: {len(prices)}개 / 실패: {len(failed)}개")


## 2. PER 계산 + IQR 이상치 제거

In [ ]:
# PER = 현재가 / EPS
df_main['PER'] = df_main['price'] / df_main['EPS']

# IQR 기준 이상치 제거 (극단적 PER 종목 필터)
Q1  = df_main['PER'].quantile(0.25)
Q3  = df_main['PER'].quantile(0.75)
IQR = Q3 - Q1
per_upper = Q3 + 1.5 * IQR

print(f"PER 상한 (IQR 기준): {per_upper:.1f}")
print(f"제거 전: {len(df_main)}개")

df_clean = df_main[(df_main['PER'] > 0) & (df_main['PER'] <= per_upper)].copy()
print(f"제거 후: {len(df_clean)}개")


## 3. 섹터별 중앙값 PER 계산

In [ ]:
# 시장 전체 중앙값 PER
market_median_per = df_clean['PER'].median()
print(f"시장 중앙값 PER: {market_median_per:.2f}")

# 섹터별 중앙값 PER
sector_median_per = df_clean.groupby('Sector')['PER'].median()
df_clean['sector_median_per'] = df_clean['Sector'].map(sector_median_per)

print("\n섹터별 중앙값 PER:")
print(sector_median_per.sort_values())


## 4. 적정주가 + 괴리율 v1 산출

In [ ]:
# 적정주가 = EPS × 적정PER
df_clean['fair_price_market'] = df_clean['EPS'] * market_median_per
df_clean['fair_price_sector'] = df_clean['EPS'] * df_clean['sector_median_per']

# 괴리율 = (현재가 - 적정주가) / 적정주가
# 음수(-) = 저평가 / 양수(+) = 고평가
df_clean['gap_market'] = (df_clean['price'] - df_clean['fair_price_market']) / df_clean['fair_price_market']
df_clean['gap_sector'] = (df_clean['price'] - df_clean['fair_price_sector']) / df_clean['fair_price_sector']

print("gap_market 분포:")
print(df_clean['gap_market'].describe())


## 5. signal 분류 + 저장

In [ ]:
# signal 분류 (v1 기준 - 성장률 조정 전)
def classify_signal(gap_market, gap_sector):
    avg = (gap_market + gap_sector) / 2
    if avg <= -0.30:
        return '강력매수'
    elif avg <= -0.15:
        return '매수'
    elif avg >= 0.30:
        return '강력매도'
    elif avg >= 0.15:
        return '매도'
    else:
        return '중립'

df_clean['signal'] = df_clean.apply(
    lambda r: classify_signal(r['gap_market'], r['gap_sector']), axis=1
)

print("signal 분포:")
print(df_clean['signal'].value_counts())

# 저장
import os
os.makedirs(f'{BASE_PATH}/data/output', exist_ok=True)
df_clean.to_csv(f'{BASE_PATH}/data/output/gap_v1_refined.csv',
                index=False, encoding='utf-8-sig')
print(f"\n✅ gap_v1_refined.csv 저장: {len(df_clean)}개")
